In [2]:
import cv2

img = cv2.imread("./image/3people.jpg")

if img is None:
    print("사진을 찾을 수 없습니다. 경로를 확인하세요.")
    exit()

# 새로 바뀜: 오른쪽 드래그용 칸 두 개 추가
state = {"drawing": False, "start": None, "cur": None, "selected": None,
         "rdrawing": False, "rstart": None}

boxes = []

def point_in_box(x, y, box):
    x1, y1, x2, y2 = box
    return x1 <= x <= x2 and y1 <= y <= y2

# 새로 추가: 7-1에서 만든 IoU 함수 그대로
def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - inter
    if union == 0:
        return 0
    return inter / union

# 새로 추가: 점으로 선택 (6단계의 오른쪽 클릭 내용을 함수로 옮김)
def select_by_point(x, y):
    state["selected"] = None
    for i, box in enumerate(boxes):
        if point_in_box(x, y, box):
            state["selected"] = i
            break
    if state["selected"] is None:
        print("선택 해제")
    else:
        print("선택: ID", state["selected"] + 1)

# 새로 추가: 드래그로 선택 (IoU 가장 높은 사람)
def select_by_drag(drag):
    best_i = None
    best_score = 0
    for i, box in enumerate(boxes):
        score = iou(drag, box)
        print("  ID", i + 1, "IoU 점수:", round(score, 2))
        if score > best_score:
            best_score = score
            best_i = i
    if best_score < 0.1:
        state["selected"] = None
        print("선택 해제 (많이 겹치는 사람 없음)")
    else:
        state["selected"] = best_i
        print("선택: ID", best_i + 1)

def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        state["drawing"] = True
        state["start"] = (x, y)
        state["cur"] = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE and state["drawing"]:
        state["cur"] = (x, y)
    elif event == cv2.EVENT_LBUTTONUP and state["drawing"]:
        state["drawing"] = False
        x0, y0 = state["start"]
        x1 = min(x0, x)
        y1 = min(y0, y)
        x2 = max(x0, x)
        y2 = max(y0, y)
        if x2 - x1 > 3 and y2 - y1 > 3:
            boxes.append((x1, y1, x2, y2))
            print("저장:", len(boxes), "번", (x1, y1, x2, y2))

    # 새로 바뀜: 오른쪽 버튼은 누르기/움직이기/떼기로 나눔
    elif event == cv2.EVENT_RBUTTONDOWN:
        state["rdrawing"] = True
        state["rstart"] = (x, y)
        state["cur"] = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE and state["rdrawing"]:
        state["cur"] = (x, y)
    elif event == cv2.EVENT_RBUTTONUP and state["rdrawing"]:
        state["rdrawing"] = False
        x0, y0 = state["rstart"]
        drag = (min(x0, x), min(y0, y), max(x0, x), max(y0, y))
        if drag[2] - drag[0] > 3 and drag[3] - drag[1] > 3:
            select_by_drag(drag)
        else:
            select_by_point(x, y)

cv2.namedWindow("bbox")
cv2.setMouseCallback("bbox", on_mouse)

while True:
    view = img.copy()

    for i, (x1, y1, x2, y2) in enumerate(boxes):
        if i == state["selected"]:
            color = (0, 0, 255)
        else:
            color = (0, 255, 0)
        cv2.rectangle(view, (x1, y1), (x2, y2), color, 2)
        cv2.putText(view, f"ID {i + 1}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    if state["drawing"]:
        cv2.rectangle(view, state["start"], state["cur"], (0, 255, 255), 1)

    # 새로 추가: 오른쪽 드래그 중이면 보라색 네모
    if state["rdrawing"]:
        cv2.rectangle(view, state["rstart"], state["cur"], (255, 0, 255), 1)

    cv2.imshow("bbox", view)

    key = cv2.waitKey(10) & 0xFF
    if key == 27:
        break
    if key == ord("c"):
        boxes.clear()
        state["selected"] = None
    if cv2.getWindowProperty("bbox", cv2.WND_PROP_VISIBLE) < 1:
        break

cv2.destroyAllWindows()

저장: 1 번 (96, 46, 176, 169)
저장: 2 번 (178, 183, 262, 305)
저장: 3 번 (264, 105, 343, 206)
  ID 1 IoU 점수: 0.0
  ID 2 IoU 점수: 0.1
  ID 3 IoU 점수: 0.18
선택: ID 3
저장: 4 번 (224, 155, 284, 266)
선택: ID 4
  ID 1 IoU 점수: 0.0
  ID 2 IoU 점수: 0.04
  ID 3 IoU 점수: 0.09
  ID 4 IoU 점수: 0.24
선택: ID 4
선택 해제
  ID 1 IoU 점수: 0.06
  ID 2 IoU 점수: 0.11
  ID 3 IoU 점수: 0.0
  ID 4 IoU 점수: 0.0
선택: ID 2
선택 해제
  ID 1 IoU 점수: 0.04
  ID 2 IoU 점수: 0.25
  ID 3 IoU 점수: 0.0
  ID 4 IoU 점수: 0.2
선택: ID 2
선택 해제
  ID 1 IoU 점수: 0.13
  ID 2 IoU 점수: 0.0
  ID 3 IoU 점수: 0.0
  ID 4 IoU 점수: 0.0
선택: ID 1
선택 해제
  ID 1 IoU 점수: 0.0
  ID 2 IoU 점수: 0.0
  ID 3 IoU 점수: 0.0
  ID 4 IoU 점수: 0.0
선택 해제 (많이 겹치는 사람 없음)
선택 해제
  ID 1 IoU 점수: 0.17
  ID 2 IoU 점수: 0.06
  ID 3 IoU 점수: 0.0
  ID 4 IoU 점수: 0.0
선택: ID 1
선택 해제
  ID 1 IoU 점수: 0.07
  ID 2 IoU 점수: 0.0
  ID 3 IoU 점수: 0.0
  ID 4 IoU 점수: 0.0
선택 해제 (많이 겹치는 사람 없음)
선택 해제
  ID 1 IoU 점수: 0.0
  ID 2 IoU 점수: 0.15
  ID 3 IoU 점수: 0.0
  ID 4 IoU 점수: 0.13
선택: ID 2
선택 해제
선택 해제
  ID 1 IoU 점수: 0.0
  ID 2 IoU 점수: 0.1
 

In [3]:
import cv2

img = cv2.imread("./image/3people.jpg")

if img is None:
    print("사진을 찾을 수 없습니다. 경로를 확인하세요.")
    exit()

# 새로 바뀜: mode(모드), sdrawing/sstart(선택용 드래그), scores(점수) 추가
state = {"drawing": False, "start": None, "cur": None, "selected": None,
         "mode": "draw", "sdrawing": False, "sstart": None, "scores": {}}

boxes = []

def point_in_box(x, y, box):
    x1, y1, x2, y2 = box
    return x1 <= x <= x2 and y1 <= y <= y2

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)
    inter = iw * ih
    area_a = (ax2 - ax1) * (ay2 - ay1)
    area_b = (bx2 - bx1) * (by2 - by1)
    union = area_a + area_b - inter
    if union == 0:
        return 0
    return inter / union

def select_by_point(x, y):
    state["scores"] = {}
    state["selected"] = None
    for i, box in enumerate(boxes):
        if point_in_box(x, y, box):
            state["selected"] = i
            break
    if state["selected"] is None:
        print("선택 해제")
    else:
        print("선택: ID", state["selected"] + 1)

def select_by_drag(drag):
    state["scores"] = {}
    best_i = None
    best_score = 0
    for i, box in enumerate(boxes):
        score = iou(drag, box)
        state["scores"][i] = score          # 새로 추가: 점수를 저장해 둠
        if score > best_score:
            best_score = score
            best_i = i
    if best_score < 0.1:
        state["selected"] = None
        print("선택 해제 (많이 겹치는 사람 없음)")
    else:
        state["selected"] = best_i
        print("선택: ID", best_i + 1, "점수:", round(best_score, 2))

def on_mouse(event, x, y, flags, param):
    # 새로 추가: 선택 모드일 때는 왼쪽 버튼으로 선택
    if state["mode"] == "select":
        if event == cv2.EVENT_LBUTTONDOWN:
            state["sdrawing"] = True
            state["sstart"] = (x, y)
            state["cur"] = (x, y)
        elif event == cv2.EVENT_MOUSEMOVE and state["sdrawing"]:
            state["cur"] = (x, y)
        elif event == cv2.EVENT_LBUTTONUP and state["sdrawing"]:
            state["sdrawing"] = False
            x0, y0 = state["sstart"]
            drag = (min(x0, x), min(y0, y), max(x0, x), max(y0, y))
            if drag[2] - drag[0] > 3 and drag[3] - drag[1] > 3:
                select_by_drag(drag)
            else:
                select_by_point(x, y)
        return

    # 여기부터는 그리기 모드 (5단계와 같음)
    if event == cv2.EVENT_LBUTTONDOWN:
        state["drawing"] = True
        state["start"] = (x, y)
        state["cur"] = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE and state["drawing"]:
        state["cur"] = (x, y)
    elif event == cv2.EVENT_LBUTTONUP and state["drawing"]:
        state["drawing"] = False
        x0, y0 = state["start"]
        x1 = min(x0, x)
        y1 = min(y0, y)
        x2 = max(x0, x)
        y2 = max(y0, y)
        if x2 - x1 > 3 and y2 - y1 > 3:
            boxes.append((x1, y1, x2, y2))
            print("저장:", len(boxes), "번", (x1, y1, x2, y2))

cv2.namedWindow("bbox")
cv2.setMouseCallback("bbox", on_mouse)

while True:
    view = img.copy()

    for i, (x1, y1, x2, y2) in enumerate(boxes):
        if i == state["selected"]:
            color = (0, 0, 255)
        else:
            color = (0, 255, 0)
        cv2.rectangle(view, (x1, y1), (x2, y2), color, 2)
        cv2.putText(view, f"ID {i + 1}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        # 새로 추가: 점수가 있으면 네모 아래에 표시
        if i in state["scores"]:
            cv2.putText(view, f"{state['scores'][i]:.2f}", (x1, y2 + 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    if state["drawing"]:
        cv2.rectangle(view, state["start"], state["cur"], (0, 255, 255), 1)
    if state["sdrawing"]:
        cv2.rectangle(view, state["sstart"], state["cur"], (255, 0, 255), 1)

    # 새로 추가: 화면 아래에 현재 모드 표시
    h = view.shape[0]
    cv2.putText(view, f"MODE: {state['mode']}  (D: draw / S: select / C: clear)",
                (10, h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.imshow("bbox", view)

    key = cv2.waitKey(10) & 0xFF
    if key == 27:
        break
    if key == ord("c"):
        boxes.clear()
        state["selected"] = None
        state["scores"] = {}
    # 새로 추가: 모드 바꾸기
    if key == ord("d"):
        state["mode"] = "draw"
        state["sdrawing"] = False
    if key == ord("s"):
        state["mode"] = "select"
        state["drawing"] = False
    if cv2.getWindowProperty("bbox", cv2.WND_PROP_VISIBLE) < 1:
        break

cv2.destroyAllWindows()

저장: 1 번 (249, 82, 328, 191)
저장: 2 번 (176, 163, 264, 278)
저장: 3 번 (102, 38, 166, 130)
선택 해제
선택: ID 2
선택: ID 3
선택: ID 1
선택: ID 2 점수: 0.18
선택: ID 3 점수: 0.34
선택: ID 1 점수: 0.27
선택: ID 2 점수: 0.22
선택 해제 (많이 겹치는 사람 없음)
선택: ID 3 점수: 0.11
선택 해제 (많이 겹치는 사람 없음)
선택: ID 2 점수: 0.12
선택: ID 2 점수: 0.1
선택: ID 1 점수: 0.32
선택: ID 2 점수: 0.46
선택: ID 2
선택: ID 1 점수: 0.48
선택: ID 2 점수: 0.29
선택 해제 (많이 겹치는 사람 없음)
선택 해제 (많이 겹치는 사람 없음)
선택 해제 (많이 겹치는 사람 없음)
선택: ID 2 점수: 0.46
선택: ID 2 점수: 0.86
선택: ID 1
선택: ID 2
선택 해제


In [4]:
python yolo_test.py

SyntaxError: invalid syntax (3856767174.py, line 1)